# KALIA prep v2 — balanced data mixture

Builds the v2 corpus on Kaggle CPU (free quota):

| Source | Share | Tokens |
|---|---|---|
| smollm fineweb-edu-dedup | 60% | 1.50B |
| TinyStories | 20% | 500M |
| Cosmopedia v2 | 15% | 375M |
| codeparrot-clean (Python) | 5% | 125M |

Outputs `/kaggle/working/data/train.bin` (~2.5B tokens, interleaved) and
`val.bin` (10M tokens, same mixture). No GPU, no secrets needed.

In [ ]:
!pip install -q tiktoken datasets

In [ ]:
import glob
import os
import shutil

work = "/kaggle/working/kalia"
if not os.path.exists(work):
    hits = sorted(glob.glob("/kaggle/input/**/prepare.py", recursive=True))
    assert hits, "attach the kalia-code-dev dataset to this notebook"
    shutil.copytree(os.path.dirname(hits[0]), work)
os.chdir(work)
print("code from", work)

In [ ]:
!mkdir -p /kaggle/working/data
!python prepare.py --source tinystories --out /kaggle/working/data/tinystories.bin --max-tokens 500000000 --meta /kaggle/working/data/meta.json
!python prepare.py --source smollm_fineweb_edu --out /kaggle/working/data/smollm_fineweb_edu.bin --max-tokens 1500000000 --meta /kaggle/working/data/meta.json
!python prepare.py --source cosmopedia --out /kaggle/working/data/cosmopedia.bin --max-tokens 375000000 --meta /kaggle/working/data/meta.json
!python prepare.py --source stack_smol --out /kaggle/working/data/stack_smol.bin --max-tokens 125000000 --meta /kaggle/working/data/meta.json
!cat /kaggle/working/data/meta.json

In [ ]:
!python mix_bins.py --shards "/kaggle/working/data/tinystories.bin:20,/kaggle/working/data/smollm_fineweb_edu.bin:60,/kaggle/working/data/cosmopedia.bin:15,/kaggle/working/data/stack_smol.bin:5" --val-tokens 10000000 --train-tokens 2400000000 --val-out /kaggle/working/data/val.bin --train-out /kaggle/working/data/train.bin --meta /kaggle/working/data/mix_meta.json
!ls -lh /kaggle/working/data/

Done. Save Version → the output `data/train.bin` + `data/val.bin` become the input for the v0.2.0 training kernel (as a kernel source, no dataset upload needed).